# D6 · Image time series and data cubes: five years of the Melamchi valley

> **Time:** about 3 hours  
> **You need:** Parts A, B and C, and Lessons D1 to D5: NDVI (D2), the flood strip of D3, reference labels (D4). From Part A: dates, `resample` and `rolling` in pandas (A11) and plots (A10). From Part C: rioxarray, `.sel(..., method="nearest")` (C7), `reproject_match` (C4) and `clip_box` (C3).  
> **By the end you can:**
> 1. open a **data cube** (x, y and time) with xarray, inspect it, and select from it by date and by place,
> 2. mask clouds with the **scene classification** (SCL), compute NDVI for every date in one line, and follow a forest and the flood-hit valley floor through five years,
> 3. summarise a time series by month (`groupby`, `resample`, `rolling`), compare the same season before and after the flood, and save results as NetCDF and GeoTIFF.

D1 to D4 worked with **two** images: 29 October 2020 and 11 November 2021. Two dates show *that* the valley floor changed. They cannot show *when*, whether the change lasted, or what a normal year looks like.

This lesson uses **112 images** of the same valley, from January 2019 to December 2023: the two clearest of each month. To keep the file small, the cells are **100 m** instead of 20 m, and there are only four layers: red (B04), near-infrared (B08), short-wave infrared (B11) and the **scene classification** (SCL), a map of what ESA's processor thinks each cell is, such as cloud or vegetation.

Many images of one place, stacked in time, are called a **data cube**. Cubes are how satellite data are handled today, from one valley to the whole planet.

*Contains modified Copernicus Sentinel data 2019–2023, processed by ESA.* The credit belongs wherever the images, or maps and charts made from them, are shown.

**Counting lines:** lines that only set up or show a plot (`fig, ax = plt.subplots()`, `plt.show()`) do not count toward the Q1/Q2/Q3 line counts.

**If results look strange:** use **Kernel → Restart Kernel and Run All Cells**. Many cells here build on earlier ones, and a variable left over from an earlier attempt can change an answer.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 39 of 39 · [Course home](../README.md)  
← Previous: [D5 · Roads and networks: what the flood cut](D05_Roads_and_networks.ipynb) · You have reached the end of the course

In [ ]:
# Optional: marks your answers as you go. Delete this cell to turn it off.
%run ../tools/tutor_on.py

## 1. A data cube: x, y and time

A single-band raster is a grid with two **dimensions**: y (rows) and x (columns), as in C1. Add a third dimension, **time**, and you have a **data cube**: the same grid, once for every date.

**xarray** (C1 used it through rioxarray) handles cubes with named dimensions. A **Dataset** holds several **data variables** that share the same dimensions, here B04, B08, B11 and SCL. Each variable on its own is a **DataArray**, the kind of object you met in Part C.

`xr.open_dataset(path, decode_coords="all")` opens a NetCDF file, a common format for cubes. `decode_coords="all"` also reads the `spatial_ref` variable, so `.rio` knows the CRS (C1).

The file stores reflectance as whole numbers (`uint16`), with 0 meaning "no data". By default xarray **masks and scales** while reading: it turns every 0 into `NaN`, and so it must store the values as decimals (`float32`). You still multiply by 0.0001 yourself (D1): this file has no scale factor that xarray could apply for you.

**Predict:** how many dimensions will `cube["B08"]` have, and how many numbers will it hold?

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import rioxarray as rxr                       # adds .rio to xarray objects (C1)
import geopandas as gpd
from rasterio.enums import Resampling
from scipy import ndimage

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)
CUBE_PATH = DATA / "s2_melamchi_timeseries_100m.nc"
S2_CREDIT = "Contains modified Copernicus Sentinel data 2019–2023, processed by ESA."

cube = xr.open_dataset(CUBE_PATH, decode_coords="all")
cube

In [ ]:
print("dimensions:", dict(cube.sizes))
print("variables: ", list(cube.data_vars))
print("B08 dims:", cube["B08"].dims, "| numbers:", cube["B08"].size)

dates = cube.indexes["time"]                   # the dates as a pandas DatetimeIndex (A11)
print("first:", dates[0], "| last:", dates[-1])
print("CRS:", cube.rio.crs, "| cell size:", cube.rio.resolution(), "| bounds:", cube.rio.bounds())

print("B08 in memory:", cube["B08"].dtype, "| on disk:", cube["B08"].encoding["dtype"], "| no-data value:", cube["B08"].encoding["_FillValue"])
raw = xr.open_dataset(CUBE_PATH, mask_and_scale=False)
print("without masking:", raw["B08"].dtype, "| cells equal to 0:", int((raw["B08"] == 0).sum()))

In [ ]:
refl = cube[["B04", "B08", "B11"]] * 0.0001     # reflectance for three variables at once (D1)
print(refl)
print("median NIR reflectance of the whole cube:", round(float(refl["B08"].median()), 3))

**Predict, answered:** `cube["B08"]` has three dimensions, `('time', 'y', 'x')`, and holds 112 × 202 × 141 = 3,189,984 numbers. Every variable has the same three dimensions. One more variable, `clear_fraction`, has only `time`: one number per date (step 3 explains it).

The first image was taken on 8 January 2019 at 04:51 UTC, the last on 3 December 2023. Each date carries a **clock time** too, the moment the satellite passed. Remember that; it matters in step 9.

The cube is in EPSG:32645 (UTM 45N, B4) with 100 m cells, so 202 rows × 141 columns cover the D3 box, about 14 km × 20 km.

In memory B08 is `float32`; on disk it is `uint16` with the no-data value 0. That is **mask and scale** at work. Opened with `mask_and_scale=False`, the raw `uint16` values come back. This cube has no 0 cells at all, but the rule would protect you if it had: a 0 counted as reflectance would pull every mean down (C5).

`cube[["B04", "B08", "B11"]] * 0.0001` multiplies three variables at once and returns a new Dataset. SCL is left out on purpose: it holds class codes, and a class code times 0.0001 means nothing. The median NIR reflectance, about 0.276, is typical of a valley that is mostly forest (D1).

**How the cube was made.** The script `../data/get_extra_data.py` built it once, online, through a **STAC** catalogue (SpatioTemporal Asset Catalog, D1). STAC is a standard way to *search* images by place, date and cloud cover, and to *load* only the bands and the area you need. Its core lines look like this. **Try this at home when you are online** (it needs the extra packages `pystac-client`, `planetary-computer` and `odc-stac`):

```python
import planetary_computer
import pystac_client
from odc.stac import load

catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                                    modifier=planetary_computer.sign_inplace)
items = catalog.search(collections=["sentinel-2-l2a"], bbox=(85.48, 27.80, 85.62, 27.98),
                       datetime="2019-01-01/2023-12-31",
                       query={"eo:cloud_cover": {"lt": 60}}).item_collection()   # search, filter by cloud
cube = load(items, bands=["B04", "B08", "B11", "SCL"], bbox=(85.48, 27.80, 85.62, 27.98),
            crs="EPSG:32645", resolution=100, groupby="solar_day")              # load onto one grid
```

The script then removed the offset of 1,000 that ESA adds to reflectance from January 2022, kept dates on which at least 30% of the box was clear, and kept the two clearest dates of each month. Open the script to read every step.

*Contains modified Copernicus Sentinel data 2019–2023, processed by ESA.*

### Practice

**Q1 · one line:** How many cells does **one** date hold? Multiply the sizes of `y` and `x` from `cube.sizes`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
cube.sizes["y"] * cube.sizes["x"]
```

28,482 cells of 100 m, about 285 km². The 20 m images of D1 to D4 had 25 times as many cells for about the same box.
</details>

**Q2 · two lines:** Print the attributes of the cube (`cube.attrs`, a dictionary) one per line, with a `for` loop over `.items()` (A4).

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for key, value in cube.attrs.items():
    print(key, ":", value)
```

Five notes travel inside the file: a title, the credit, how to turn the numbers into reflectance, how the dates were chosen, and what the SCL codes mean. Attributes make a file explain itself; read them first whenever you open a new cube.
</details>

**Q3 · three lines:** Print the median reflectance of each of B04, B08 and B11 over the whole cube, rounded to 3 decimals, with a loop over `refl.data_vars`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for name in refl.data_vars:
    median_refl = float(refl[name].median())
    print(name, round(median_refl, 3))
```

About 0.056 for red, 0.276 for near-infrared and 0.203 for short-wave infrared. Red is low and NIR high: the spectral signature of green vegetation (D1), because most of the valley is forest. `float(...)` turns a one-number DataArray into a plain Python number.
</details>

## 2. Selecting by date and by place

`.sel()` selects by **label**, as in C7. On a cube it works on time too:

- `cube.sel(time="2021-11")`: every date in November 2021. A shorter date string selects a whole month or year, like slicing a DatetimeIndex in A11.
- `cube.sel(time=slice("2021-01", "2021-06"))`: everything from January to June 2021, both ends included.
- `cube.sel(time="2021-11-12", method="nearest")`: the **one** date closest to 12 November 2021. The `time` dimension disappears, because one date is left.
- `cube.sel(x=356550, y=3084850, method="nearest")`: the cell nearest to a point in metres (UTM 45N). What is left is a **time series**: one value per date.

`.isel()` selects by **position** instead, like `.iloc` in pandas (A9): `cube.isel(time=0)` is the first date.

**Predict:** how many dates will `cube.sel(time="2021-11")` hold, and which dimensions will `cube["B08"].sel(x=356550, y=3084850, method="nearest")` keep?

In [ ]:
nov_2021 = cube.sel(time="2021-11")
print("November 2021:", nov_2021.sizes["time"], "dates:", list(nov_2021.indexes["time"].date))

one_date = cube.sel(time="2021-11-12", method="nearest")
print("nearest to 12 Nov 2021:", one_date["time"].values, "| B08 dims:", one_date["B08"].dims)

first_half = cube.sel(time=slice("2021-01", "2021-06"))
print("January to June 2021:", first_half.sizes["time"], "dates")

cell_nir = refl["B08"].sel(x=356550, y=3084850, method="nearest")
print("one cell:", cell_nir.dims, cell_nir.sizes["time"], "values, at x =", float(cell_nir.x), "y =", float(cell_nir.y))

In [ ]:
left, bottom, right, top = cube.rio.bounds()
extent_km = [left / 1000, right / 1000, bottom / 1000, top / 1000]
SWIR_COMPOSITE = ["B11", "B08", "B04"]          # bare ground and sediment stand out (D1)


def composite(image, band_names, tops):
    """Three variables of one date as an RGB picture, each divided by its top value and clipped to 0-1 (D1)."""
    rgb = np.dstack([image[name].values for name in band_names])
    return np.clip(np.nan_to_num(rgb / tops), 0, 1)


day_before = refl.sel(time="2020-10-29", method="nearest")
day_after = refl.sel(time="2021-11-11", method="nearest")
tops = np.nanpercentile(np.dstack([day_before[b].values for b in SWIR_COMPOSITE]), 98, axis=(0, 1))   # one stretch for both

fig, axes = plt.subplots(1, 2, figsize=(11, 8))   # fig: the whole picture; axes: its two boxes to draw in
for ax, image in zip(axes, [day_before, day_after]):
    ax.imshow(composite(image, SWIR_COMPOSITE, tops), extent=extent_km)
    ax.set_title(str(image["time"].values)[:10] + ", SWIR-NIR-red, 100 m")
    ax.set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
fig.text(0.01, 0.02, S2_CREDIT, fontsize=9)
plt.show()

**Predict, answered:** November 2021 holds 2 dates, the 11th and the 13th. The cell keeps only `('time',)`: 112 values, one per date, for the cell centred at x = 356,550 m and y = 3,084,850 m, on the valley floor.

`method="nearest"` found 11 November 2021 for "12 November", and the `time` dimension is gone from B08: one date is a plain map, `('y', 'x')`. January to June 2021 holds 12 dates.

The two pictures are the dates of D1 to D4, now from the cube. `fig` is the whole picture; `axes` holds its two boxes, and each `ax` in the loop is one box to draw in. In the SWIR–NIR–red composite, vegetation is green and bare ground, gravel and sediment are pink to violet. In 2020 the river is a thin violet line. In 2021 a **wide violet strip** follows the valley floor from north to south: the flood strip of D3, coarser but clear even at 100 m.

`composite` is D1's function, changed to take variables of a Dataset instead of bands.

*Contains modified Copernicus Sentinel data 2019–2023, processed by ESA.*

### Practice

**Q1 · one line:** How many dates does the cube hold for the year **2021**? Use `.sel` with a year string and `.sizes`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
cube.sel(time="2021").sizes["time"]
```

21 dates, not 24: two clear dates per month except in the monsoon. Step 3 shows which months are missing.
</details>

**Q2 · two lines:** Which image is nearest to **15 June 2021**, and what share of the box was clear on it? Select with `method="nearest"`, then print its `time` and its `clear_fraction` rounded to 2 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
june = cube.sel(time="2021-06-15", method="nearest")
print(june["time"].values, round(float(june["clear_fraction"]), 2))
```

6 June 2021, with about 0.58 of the box clear. The nearest image is more than a week earlier: `method="nearest"` always returns a date, however far away it is, so print it and check.
</details>

**Q3 · three lines:** Melamchi town lies at 85.5761° E, 27.8293° N (from OpenStreetMap, D5). Turn the point into UTM 45N with GeoPandas (B4), and print the NIR reflectance of the nearest cell on the date nearest to 11 November 2021, rounded to 3 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
town = gpd.GeoSeries(gpd.points_from_xy([85.5761], [27.8293]), crs="EPSG:4326").to_crs("EPSG:32645")
town_nir = refl["B08"].sel(x=town.x.iloc[0], y=town.y.iloc[0], time="2021-11-11", method="nearest")
print(round(town.x.iloc[0]), round(town.y.iloc[0]), round(float(town_nir), 3))
```

The town lies at about x = 359,770 m, y = 3,079,107 m, and its cell reflects about 0.2 in the NIR, well below the cube's median of 0.276: at 100 m it mixes houses, roads and the river bed. `method="nearest"` applies to all three labels at once: x, y and time.
</details>

## 3. Clouds and the scene classification (SCL)

Even the clearest dates have some clouds. A cloud is bright in every band, so it would ruin NDVI wherever it sits. You must **mask** it, cell by cell and date by date.

Level-2A images come with a **scene classification layer** (SCL): ESA's processor labels every cell with a code. The codes are: 0 no data, 1 saturated or defective, 2 dark area or topographic shadow, 3 cloud shadow, **4 vegetation**, **5 not vegetated**, **6 water**, 7 unclassified, 8 cloud (medium probability), 9 cloud (high probability), 10 thin cirrus, 11 snow or ice.

This lesson keeps **4, 5 and 6**: the codes that mean "the ground is visible". It drops 11 too, although the file's own `clear_fraction` counted snow as clear. NDVI on snow says nothing about plants, and the example shows a second reason.

`cube["SCL"].isin([4, 5, 6])` gives a True/False cube; `.mean(dim=["x", "y"])` turns it into the **share of clear cells** of each date. SCL is itself the output of a model. Step 4 and review 1 show that it can be wrong.

In [ ]:
codes, n_cells = np.unique(cube["SCL"].values, return_counts=True)
scl_table = pd.DataFrame({"code": codes.astype(int), "cells": n_cells})
scl_table["percent"] = (scl_table["cells"] / scl_table["cells"].sum() * 100).round(2)
print(scl_table.to_string(index=False))

snow_cells = (cube["SCL"] == 11).sum(dim=["x", "y"]).to_pandas()      # a pandas Series, one value per date
print("cells called snow, by month:", snow_cells.groupby(snow_cells.index.month).sum().to_dict())

In [ ]:
clear = cube["SCL"].isin([4, 5, 6])                  # True where the ground is visible
clear_share = clear.mean(dim=["x", "y"])                # one number per date
print("clear share: lowest", round(float(clear_share.min()), 2), "| median", round(float(clear_share.median()), 2))

per_month = pd.Series(dates.month).value_counts().sort_index()
print("dates per month, all years:", per_month.to_dict())
print(pd.crosstab(dates.year, dates.month))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [2, 1]})
axes[0].plot(clear_share["time"], clear_share * 100, marker=".", linestyle="none", color="steelblue")
axes[0].set_ylabel("Clear cells (%)")
axes[0].set_title("Share of the box with SCL 4, 5 or 6, on each date")
axes[1].bar(per_month.index, per_month.values, color="grey")
axes[1].set_xticks(range(1, 13))
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Dates in 2019-2023")
axes[1].set_title("Dates per month (2 per year at most)")
fig.text(0.01, -0.02, S2_CREDIT, fontsize=9)
plt.show()

**The codes.** Code 4, vegetation, covers about 72.1% of all cells on all dates, and code 5, not vegetated, about 14.2%. High-probability cloud (9) covers about 7.4%, medium-probability cloud (8) about 1.9%. A few hundred cells carry codes 12 to 15, which are not in ESA's list; they are not 4, 5 or 6, so the mask drops them anyway.

**Snow.** Most cells called snow come in **September (884), June (440) and August (404)**: the monsoon. Real snow on the ridges is likely in winter, and February has 332. The monsoon "snow" is probably bright cloud that the processor mistook for snow: a second reason to drop code 11.

**The monsoon gap.** The median date is 97% clear, but the worst only 32%, and the scatter plot shows the cloudy dates gathering in the middle of each year. The bars show the real gap: **July has only 5 dates** in five years, August 8 and September 9, against 10 for every other month. The table shows the holes: no image at all for **August 2021** and **July 2023**. The script asked for the two clearest dates of each month; in a Himalayan monsoon, some months have none that are clear enough.

`.to_pandas()` turns a DataArray with one dimension into a pandas Series (A9), so the pandas tools of A11 work on it. `pd.crosstab` counts the dates for every year and month (D4). `gridspec_kw={"width_ratios": [2, 1]}` makes the left box twice as wide as the right one.

*Contains modified Copernicus Sentinel data 2019–2023, processed by ESA.*

### Practice

**Q1 · one line:** How many dates have **less than 70%** of the box clear? Use `clear_share` and `.sum()`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
int((clear_share < 0.7).sum())
```

18 dates. Each of them still counts, but whole parts of the valley are missing on them: a median over the box on such a date describes only the parts that were clear.
</details>

**Q2 · two lines:** Which month has the lowest **mean** clear share? Turn `clear_share` into a pandas Series with `.to_pandas()`, then group it by `.index.month` (A11) and print the means rounded to 2 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
share_series = clear_share.to_pandas()
print(share_series.groupby(share_series.index.month).mean().round(2))
```

June and August, both about 0.67, then July (0.69) and September (0.73); October to April are all 0.95 or more. Even the dates the script chose as the clearest of their month are cloudier in the monsoon, and there are fewer of them.
</details>

**Q3 · three lines:** The file's `clear_fraction` counted code 11 (snow) as clear; your `clear_share` does not. Find the date where the two differ most: compute the difference, then use `.argmax()` with `.isel(time=...)`, and print the date and the difference in percentage points.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
difference = cube["clear_fraction"] - clear_share
biggest = int(difference.argmax(dim="time"))
print(difference["time"].values[biggest], round(float(difference.isel(time=biggest)) * 100, 2))
```

27 September 2021, by about 2.38 percentage points: on that monsoon date the processor called hundreds of cells snow, which the file counted as clear and you did not. `.argmax(dim="time")` gives the position of the largest value along `time`, as `np.argmax` does in A8.
</details>

## 4. NDVI for every date: a forest and the valley floor

NDVI is (NIR − red) / (NIR + red), as in D2. xarray applies arithmetic to **every date at once**, so one line gives an NDVI cube. `.where(clear)` then turns every cell that is not clear into `NaN`.

To follow an **area** through time, reduce each date to one number: `.median(dim=["x", "y"])`. The median ignores `NaN` and resists the few cloudy cells that SCL missed.

Two areas to compare:

- **A forest box**: 1 km × 1 km at x 358.5–359.5 km, y 3,093–3,094 km. The example checks that ESA WorldCover (D4) calls it tree cover.
- **The flood-hit valley floor**: D3's main strip, rebuilt here from the two 20 m images with D3's rule in short (NDVI fall below −0.3, the largest connected group after joining near neighbours). The DEM rule of D3 is left out: the main strip lies on the valley floor anyway (D3 step 6). `reproject_match` with `Resampling.average` (C4) gives each 100 m cell the **share** of its area that the strip covers. Cells at least half covered count as valley floor.

**Predict:** what will the valley floor's NDVI do in 2021? And when exactly, given the dates the cube has?

In [ ]:
ndvi_all = (refl["B08"] - refl["B04"]) / (refl["B08"] + refl["B04"])   # 112 NDVI maps in one line
ndvi = ndvi_all.where(clear)                                                 # NaN where SCL is not 4, 5 or 6
print(ndvi.dims, ndvi.shape, "| clear values:", int(ndvi.count()), "of", ndvi.size)

six_dates = ndvi.sel(time="2021").isel(time=slice(0, None, 4))             # every 4th date of 2021
fig, axes = plt.subplots(2, 3, figsize=(11, 9.5), sharex=True, sharey=True)
for ax, date in zip(axes.flat, six_dates["time"].values):
    im = ax.imshow(six_dates.sel(time=date), extent=extent_km, cmap="YlGn", vmin=0, vmax=0.9)
    ax.set_title(str(date)[:10])
fig.colorbar(im, ax=axes, shrink=0.6, label="NDVI (white: masked by SCL)")
fig.suptitle("NDVI through 2021, 100 m")
fig.text(0.01, 0.02, S2_CREDIT, fontsize=9)
plt.show()

In [ ]:
def ndvi_20m(path):
    """NDVI of a Part D 20 m image (D2), as a DataArray on its own grid."""
    image = rxr.open_rasterio(path, masked=True)
    image = image.assign_coords(band=list(image.attrs["long_name"]))
    image.attrs = {}
    red, nir = image.sel(band="B04"), image.sel(band="B08")
    return (nir - red) / (nir + red)


d_ndvi_20m = ndvi_20m(DATA / "s2_melamchi_after.tif") - ndvi_20m(DATA / "s2_melamchi_before.tif")
change_20m = (d_ndvi_20m < -0.3).values
EIGHT = np.ones((3, 3), dtype=bool)                                          # D3 step 4
groups, n_groups = ndimage.label(ndimage.binary_dilation(change_20m, structure=EIGHT), structure=EIGHT)
groups = np.where(change_20m, groups, 0)
ids, counts = np.unique(groups[groups > 0], return_counts=True)
strip_20m = d_ndvi_20m.copy(data=(groups == ids[counts.argmax()]).astype("float32"))   # 1 in the strip, 0 elsewhere
print("D3's main strip:", int(strip_20m.sum()), "cells of 20 m =", round(float(strip_20m.sum()) * 0.0004, 2), "km²")

strip_share = strip_20m.rio.reproject_match(cube["B08"], resampling=Resampling.average)   # share of each 100 m cell
floor = strip_share >= 0.5
print("valley-floor cells of 100 m:", int(floor.sum()), "=", round(int(floor.sum()) * 0.01, 2), "km²")

wc_box = rxr.open_rasterio(DATA / "worldcover_melamchi_2021.tif").rio.clip_box(358500, 3093000, 359500, 3094000)
print("forest box, WorldCover tree cover:", round(float((wc_box == 10).mean()) * 100, 1), "%")

In [ ]:
forest_ndvi = ndvi.sel(x=slice(358500, 359500), y=slice(3094000, 3093000)).median(dim=["x", "y"])   # y: north first
floor_ndvi = ndvi.where(floor).median(dim=["x", "y"])
print(floor_ndvi.sel(time=slice("2021-05", "2021-09")).to_pandas().round(3))

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(forest_ndvi["time"], forest_ndvi, marker=".", color="darkgreen", label="forest box (1 km²)")
ax.plot(floor_ndvi["time"], floor_ndvi, marker=".", color="saddlebrown", label="valley floor (D3's strip)")
ax.set_ylabel("Median NDVI of clear cells")
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("NDVI of two areas of the Melamchi valley, 2019-2023")
fig.text(0.01, -0.02, S2_CREDIT, fontsize=9)
plt.show()

`ndvi` has the same three dimensions as the cube: 2,807,264 of its 3,189,984 values are clear; the rest are `NaN`. On the six maps, white means masked. On 7 January only small specks are white, probably shadows (SCL codes 2 and 3). On 15 May clouds hide parts of the north, and on 29 July, in the monsoon, large parts of the valley. The flood strip appears on 29 July as a pale line where the clouds allow, and it is plain on 14 October and 16 December. Even there, a few white cells sit **inside** the strip: step 4's Q3 and review 1 explain them.

The rebuilt strip has 7,727 cells of 20 m, about **3.09 km²**: the same largest group as in D3 step 4. On the 100 m grid, **283 cells** (2.83 km²) are at least half strip. They are the valley floor of this lesson. WorldCover calls 99.9% of the forest box tree cover.

**Predict, answered.** The forest stays near 0.8 on most dates; its few deep dips fall in the monsoon (step 5 looks at them). The valley floor moves with the seasons, then **falls** in 2021 and never comes back. The printed dates pin it down: about 0.552 on 6 June 2021, then about 0.056 on **29 July 2021**, the next date in the cube. The drop happened between those two dates, which fits the June 2021 flood. No image can give the day: the monsoon hid the valley for almost two months.

One honest warning: the strip was **chosen** because its NDVI fell between October 2020 and November 2021. That it falls in this chart is no discovery. What the chart adds is everything else: the stable years before, the **timing**, and what happened afterwards.

`y=slice(3094000, 3093000)` puts the **northern** edge first, because `y` runs from north to south in this cube, like rows in an image (C1). Step 9 shows what happens the other way round.

*Contains modified Copernicus Sentinel data 2019–2023, processed by ESA.*
*Forest check: © ESA WorldCover project 2021 / Contains modified Copernicus Sentinel data (2021) processed by ESA WorldCover consortium (CC BY 4.0).*

### Practice

**Q1 · one line:** What is the median NDVI of the **forest box** over all five years? Use `float()` and round to 2 decimals.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(float(forest_ndvi.median()), 2)
```

About 0.80. A median over 112 dates is one number for the whole cube; most of the forest's dates sit close to it.
</details>

**Q2 · two lines:** Select the NDVI series of the single cell nearest to x = 356,550 m, y = 3,084,850 m (step 2), and print its median NDVI for **2020** and for **2022**, rounded to 2 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
cell_ndvi = ndvi.sel(x=356550, y=3084850, method="nearest")
print(round(float(cell_ndvi.sel(time="2020").median()), 2), round(float(cell_ndvi.sel(time="2022").median()), 2))
```

About 0.53 in 2020 and 0.06 in 2022. One cell tells the same story as the whole valley floor. A single cell is noisier than the median of 283, so always check whether one cell is typical before you trust it.
</details>

**Q3 · three lines:** How many clear cells does the valley floor have on each date? Count them with `.count(dim=["x", "y"])` on `ndvi.where(floor)`, then print the mean count for 2019–2020 and for July 2021 to January 2022, rounded to 0 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
floor_count = ndvi.where(floor).count(dim=["x", "y"])
print(round(float(floor_count.sel(time=slice("2019", "2020")).mean())))
print(round(float(floor_count.sel(time=slice("2021-07", "2022-01")).mean())))
```

About 272 clear cells (of 283) per date before, but only about 112 after the flood. The sky was not cloudier; review 1 shows that SCL labelled the fresh sediment itself as cloud or unclassified. The mask hides part of the very change you are looking for.
</details>

## 5. The seasonal cycle: groupby month

Plants follow the seasons. To see the **typical year**, put all Januaries together, all Februaries together, and so on, then take the median of each group. In xarray:

`series.groupby("time.month").median()`

`"time.month"` is the month number of each date (1 to 12), like `.dt.month` in pandas (A11). The result has a new dimension, `month`, instead of `time`. A typical year built this way is called a **climatology** (A11).

Monsoon months are **noisy**: fewer dates, and on each date fewer clear cells. Thin cloud and haze that SCL misses lower NDVI, so monsoon values scatter more. `.std()` per month measures that scatter, and `.count()` shows how many dates stand behind each median.

For the valley floor, use only the dates **before** the flood (up to May 2021). A typical year that mixes before and after would describe neither.

In [ ]:
forest_by_month = forest_ndvi.groupby("time.month").median()
floor_by_month = floor_ndvi.sel(time=slice("2019-01", "2021-05")).groupby("time.month").median()
season = pd.DataFrame({
    "forest_median": forest_by_month.to_pandas(),
    "forest_std": forest_ndvi.groupby("time.month").std().to_pandas(),
    "forest_dates": forest_ndvi.groupby("time.month").count().to_pandas(),
    "floor_median_before": floor_by_month.to_pandas(),
})
print(season.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(season.index, season["forest_median"], marker="o", color="darkgreen", label="forest box, 2019-2023")
ax.fill_between(season.index, season["forest_median"] - season["forest_std"], season["forest_median"] + season["forest_std"],
                color="darkgreen", alpha=0.15, label="forest: median ± one standard deviation")
ax.plot(season.index, season["floor_median_before"], marker="o", color="saddlebrown", label="valley floor, Jan 2019-May 2021")
ax.axvspan(6, 9, color="lightblue", alpha=0.4, label="monsoon months (June-September)")
ax.set_xticks(range(1, 13))
ax.set_xlabel("Month")
ax.set_ylabel("Median NDVI")
ax.set_ylim(0, 1)
ax.legend(loc="lower left", fontsize=8)
ax.set_title("The typical year (climatology) of two areas")
fig.text(0.01, -0.03, S2_CREDIT, fontsize=9)
plt.show()

The forest is green all year: its monthly medians lie between about 0.697 (April) and 0.865 (October). Its scatter is small from October to January (standard deviation 0.008 to 0.022) and largest in the monsoon, up to about 0.204 in September. July's median of 0.833 rests on only 4 dates, so it says little.

The valley floor, before the flood, has a clear **seasonal cycle**: lowest from November to April (about 0.44 to 0.51), highest at the end of the monsoon, in September and October (about 0.647 and 0.643). Fields and grass on the valley floor green up with the monsoon rain and are harvested or dry by winter.

That matters for change detection. Comparing a valley-floor image from December with one from September would show a difference of about 0.2 in NDVI (Q2) with no flood at all. Step 7 therefore compares **the same months** in different years.

`fill_between` shades the area between two lines, here one standard deviation either side of the median. `axvspan(6, 9)` shades a band of the x axis.

*Contains modified Copernicus Sentinel data 2019–2023, processed by ESA.*

### Practice

**Q1 · one line:** In which month is the forest **greenest**? Use `.idxmax()` on `forest_by_month`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
int(forest_by_month.idxmax())
```

Month 10, October: right after the monsoon, with clear skies and fully grown leaves. `.idxmax()` returns the **label** of the largest value (here the month number), not its position like `.argmax()`.
</details>

**Q2 · two lines:** How much does the valley floor's typical NDVI swing over the year before the flood? Print the largest and smallest values of `floor_by_month` and their difference, rounded to 2 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
high, low = float(floor_by_month.max()), float(floor_by_month.min())
print(round(high, 2), round(low, 2), round(high - low, 2))
```

From about 0.65 to 0.44: a swing of about 0.21, two thirds of D3's threshold of 0.3. Season alone can look like change.
</details>

**Q3 · three lines:** Compute the **anomaly** of every forest date: its NDVI minus the typical value of its month. In xarray, `forest_ndvi.groupby("time.month") - forest_by_month` does it in one line. Print the three most negative anomalies with their dates (`.to_pandas()`, `.nsmallest(3)`), rounded to 3 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
forest_anomaly = forest_ndvi.groupby("time.month") - forest_by_month
print(forest_anomaly.to_pandas().nsmallest(3).round(3))
```

27 September 2020 (about −0.601), 16 July 2020 (−0.380) and 28 August 2023 (−0.317). All three are monsoon dates: thin cloud or haze that SCL let through, not a forest that changed and recovered within weeks. An anomaly is only as good as the data and the typical year behind it.
</details>

## 6. Monthly steps, smoothing and gaps

The cube's dates are irregular: two per month, at different days, and none in some months. Many methods want **regular** steps.

- `series.resample(time="1MS").median()` puts the dates into calendar months (`"1MS"`: one **m**onth, labelled by its **s**tart) and takes the median of each, as in A11. A month without any date becomes `NaN`. On the whole cube, the same line makes a **monthly cube**: 60 maps.
- `.rolling(time=3, center=True, min_periods=2).mean()` is a **moving average** over three months (A11): each month becomes the mean of itself and its two neighbours, if at least two of the three have a value.
- `.interpolate_na(dim="time", limit=1)` fills a gap of at most one month with a straight line between its neighbours.

All three **change the data**. Smoothing hides noise, but it also blurs sudden events. Interpolation invents values that no satellite measured. Use them knowingly, and say that you did.

**Predict:** the flood dropped NDVI from one image to the next. What will a 3-month moving average do to that drop?

In [ ]:
floor_monthly = floor_ndvi.resample(time="1MS").median()
missing = floor_monthly["time"].values[floor_monthly.isnull().values]
print("months:", floor_monthly.sizes["time"], "| without a value:", [str(month)[:7] for month in missing])

floor_smooth = floor_monthly.rolling(time=3, center=True, min_periods=2).mean()
floor_filled = floor_monthly.interpolate_na(dim="time", limit=1)
around_flood = pd.DataFrame({"monthly": floor_monthly.to_pandas(), "smoothed": floor_smooth.to_pandas(),
                             "filled": floor_filled.to_pandas()}).loc["2021-03":"2021-10"]
print(around_flood.round(3))

ndvi_monthly = ndvi.resample(time="1MS").median()          # the whole cube: one map per month
print("monthly cube:", dict(ndvi_monthly.sizes))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(floor_ndvi["time"], floor_ndvi, ".", color="lightgrey", label="each date")
ax.plot(floor_monthly["time"], floor_monthly, marker="o", markersize=4, color="saddlebrown", label="monthly median")
ax.plot(floor_smooth["time"], floor_smooth, color="black", linewidth=2, label="3-month moving average")
ax.set_ylabel("Median NDVI, valley floor")
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("Valley floor: dates, monthly medians and smoothing")
fig.text(0.01, -0.02, S2_CREDIT, fontsize=9)
plt.show()

Five years make **60 months**. Two of them have no valley-floor value: August 2021 and July 2023, the months without any image (step 3). The brown line breaks there; Matplotlib does not draw across `NaN`.

**Predict, answered.** The monthly median of the valley floor falls from about 0.555 in June 2021 to about 0.056 in July. The moving average spreads that step over three months: it is already down to about 0.367 in **June**, before the flood, and still about 0.305 in July. Smoothing moved part of the drop into the wrong month. For a slow change, such as a forest regrowing, a moving average helps; for a sudden event it hides the date.

`interpolate_na(limit=1)` filled August 2021 with about 0.100, halfway between July and September. That looks harmless here, but no satellite saw the valley in August 2021. A filled value must never be reported as an observation.

The monthly cube of the whole valley has 60 dates instead of 112: `resample` works on maps exactly as on one series.

*Contains modified Copernicus Sentinel data 2019–2023, processed by ESA.*

### Practice

**Q1 · one line:** How many months of the **monthly cube** `ndvi_monthly` have no clear value at all for the cell nearest to x = 356,550 m, y = 3,084,850 m?

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
int(ndvi_monthly.sel(x=356550, y=3084850, method="nearest").isnull().sum())
```

5 months: the two months without any image, plus three months in which every image was masked over that cell. One cell has more gaps than the median of a whole area.
</details>

**Q2 · two lines:** Make the **forest** box monthly too, and print the number of months without a value and the mean of the monthly medians for 2019, rounded to 3 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
forest_monthly = forest_ndvi.resample(time="1MS").median()
print(int(forest_monthly.isnull().sum()), round(float(forest_monthly.sel(time="2019").mean()), 3))
```

4 months without a value, two more than the valley floor: on some dates clouds covered the whole forest box. In 2019 the monthly medians average about 0.797.
</details>

**Q3 · three lines:** Interpolation can hide how long a gap really was. Fill `floor_monthly` **without** a limit (`interpolate_na(dim="time")`), and print how many values were invented in each case: no limit and `limit=1`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
filled_all = floor_monthly.interpolate_na(dim="time")
invented = floor_monthly.isnull() & filled_all.notnull()
print(int(invented.sum()), int((floor_monthly.isnull() & floor_filled.notnull()).sum()))
```

2 and 2. Here every gap is one month long, so the limit makes no difference. With a longer gap, `limit=1` would leave the inner months empty, which is the honest choice. `.notnull()` is the opposite of `.isnull()`.
</details>

## 7. Before and after: the same season, five years

Step 5 showed that the valley floor's NDVI swings with the seasons. So compare **the same months** in different years. October to December is the best season here: the monsoon is over, the sky is clear and the fields are still green.

`ndvi.sel(time=ndvi["time.month"] >= 10)` keeps the October to December dates of every year. `.groupby("time.year").median()` then gives **one map per year**: the median NDVI of each cell over that year's October to December.

Subtracting two of those maps gives a **change map**, as in D3, but now each side is the median of about six images instead of one. Medians of several dates resist single bad images: a cloud on one date, a shadow on another.

Be honest about the limits. This is **one** valley, and a 100 m cell mixes river, bank and fields: the strip is often only one or two 100 m cells wide. And the valley floor was chosen from D3's change, so 2020 against 2021 cannot be an independent test. The years 2019, 2022 and 2023 are new information.

In [ ]:
oct_dec = ndvi.sel(time=ndvi["time.month"] >= 10)
yearly = oct_dec.groupby("time.year").median()                    # one map per year
print(dict(yearly.sizes), "| dates per year:", oct_dec.groupby("time.year").count(dim="time").max(dim=["x", "y"]).values)

by_year = pd.DataFrame({
    "valley_floor": yearly.where(floor).median(dim=["x", "y"]).to_pandas(),
    "forest_box": yearly.sel(x=slice(358500, 359500), y=slice(3094000, 3093000)).median(dim=["x", "y"]).to_pandas(),
    "whole_box": yearly.median(dim=["x", "y"]).to_pandas(),
})
print(by_year.round(3))

In [ ]:
change_map = yearly.sel(year=2021) - yearly.sel(year=2020)        # Oct-Dec 2021 minus Oct-Dec 2020
fell = change_map < -0.3
print("cells with a fall below -0.3:", int(fell.sum()), "=", round(int(fell.sum()) * 0.01, 2), "km²")
print("of them on the valley floor:", int((fell & floor).sum()), "| touching the strip:", int((fell & (strip_share > 0)).sum()),
      "| away from it:", int((fell & (strip_share == 0)).sum()))
print("valley-floor cells without a value:", int((floor & change_map.isnull()).sum()))

fig, axes = plt.subplots(1, 2, figsize=(12, 8))
im = axes[0].imshow(change_map, extent=extent_km, cmap="RdBu", vmin=-0.6, vmax=0.6)
axes[0].set_title("NDVI, Oct-Dec 2021 minus Oct-Dec 2020")
fig.colorbar(im, ax=axes[0], shrink=0.6, label="Change in NDVI (red = fall)")
axes[1].imshow(change_map, extent=extent_km, cmap="RdBu", vmin=-0.6, vmax=0.6)
axes[1].contour(cube["x"] / 1000, cube["y"] / 1000, floor, levels=[0.5], colors="black", linewidths=0.8)
axes[1].set_xlim(354, 362)
axes[1].set_ylim(3076, 3090)
axes[1].set_title("Zoom, with the valley floor outlined")
for ax in axes:
    ax.set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
fig.text(0.01, 0.02, S2_CREDIT, fontsize=9)
plt.show()

Every year has up to 6 October–December dates per cell. The table is the core result of this lesson:

- **2019 and 2020**: the valley floor sits at about 0.498 and 0.477. Two pre-flood years agree: the before state was stable.
- **2021**: about 0.070. The fall after the flood.
- **2022 and 2023**: about 0.183 and 0.112. No recovery towards the old values in the two years after the flood. In 2022 NDVI rose a little and in 2023 it fell again; the cube cannot say why. Plants returning, new sediment, a river changing its course or construction would all change NDVI.
- The forest box stays between about 0.840 and 0.865, and the whole box between about 0.710 and 0.765: the fall belongs to the valley floor, not to the whole image.

The change map shows a red ribbon along the river and pale noise elsewhere. 242 cells (2.42 km²) fell by more than 0.3; 150 of them are valley-floor cells, 210 touch D3's strip, and only 32 lie away from it. A median of several dates resists single bad images, so few specks remain on the hillsides.

The white cells **inside** the outline have no value: 73 of the 283 valley-floor cells had no clear date at all in October–December 2021, although the sky was clear on most of those dates (step 3). Review 1 shows why. The 2021 median of the valley floor rests on the other cells.

`contour(x, y, grid, levels=[0.5])` draws the line where the grid crosses 0.5: here the edge of the True/False valley floor.

*Contains modified Copernicus Sentinel data 2019–2023, processed by ESA.*

### Practice

**Q1 · one line:** What is the median change of the **whole** change map? Use `float()` and round to 3 decimals.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(float(change_map.median()), 3)
```

About 0.001: almost nothing. Most of the valley did not change between the two autumns, as in D3.
</details>

**Q2 · two lines:** Is the valley floor still down in 2023? Make a second change map, Oct–Dec 2023 minus Oct–Dec 2020, and print the median of its valley-floor cells, rounded to 3 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
change_2023 = yearly.sel(year=2023) - yearly.sel(year=2020)
print(round(float(change_2023.where(floor).median()), 3))
```

About −0.35: two years later, the valley floor is still far below its 2020 NDVI.
</details>

**Q3 · three lines:** Did the forest box change at all? Print, for each year in `yearly`, the median NDVI of the forest box minus that of 2019, rounded to 3 decimals, in a loop over `yearly["year"].values`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
forest_yearly = yearly.sel(x=slice(358500, 359500), y=slice(3094000, 3093000)).median(dim=["x", "y"])
for year in yearly["year"].values:
    print(year, round(float(forest_yearly.sel(year=year) - forest_yearly.sel(year=2019)), 3))
```

0.0, 0.025, 0.02, 0.008 and 0.011: all within about 0.025 of 2019. Differences this small are the noise of the method (different dates, light, haze), not change in the forest.
</details>

## 8. Big cubes with dask, and saving results

This cube fits in memory: about 51 MB. A cube of the whole of Nepal at 10 m, with every image of five years, would not fit in any laptop. **dask** solves that. Open the file with `chunks=`, and xarray reads nothing yet. Each variable becomes a **dask array**: a list of **chunks** (blocks of the data) and a **recipe** of what to do with them. This is **lazy loading**.

Every calculation on a dask array only adds to the recipe. `.compute()` finally runs it, one chunk at a time, on all processor cores, and returns ordinary NumPy values. With `chunks={}` xarray uses the blocks stored in the file; `.chunk({"time": 28})` cuts them into pieces of 28 dates.

Then save your results:

- **NetCDF** keeps a cube with all its dimensions, coordinates and the CRS: `yearly.to_netcdf(path)`.
- **GeoTIFF** (C1) holds maps without a time dimension, for QGIS or any GIS: `.rio.to_raster(path)` on one 2-D map.

In [ ]:
lazy = xr.open_dataset(CUBE_PATH, decode_coords="all", chunks={}).chunk({"time": 28})
print(type(lazy["B08"].data))
print("chunks along time:", lazy["B08"].chunks[0])

lazy_ndvi = (lazy["B08"] - lazy["B04"]) / (lazy["B08"] + lazy["B04"])     # the 0.0001 cancels out in NDVI
lazy_mean = lazy_ndvi.where(lazy["SCL"].isin([4, 5, 6])).mean(dim="time")
print("before compute:", type(lazy_mean.data))

mean_map = lazy_mean.compute()                                           # now dask reads and calculates
print("after compute: ", type(mean_map.data))
print("same as without dask:", bool(np.allclose(mean_map, ndvi.mean(dim="time"), equal_nan=True)))

In [ ]:
yearly.name = "ndvi_oct_dec_median"
yearly.attrs = {"description": "Median NDVI of clear dates (SCL 4, 5, 6), October to December of each year",
                "credit": S2_CREDIT}
nc_path = OUT / "d06_ndvi_oct_dec_by_year.nc"
yearly.to_netcdf(nc_path)

tif_path = OUT / "d06_ndvi_change_2020_2021.tif"
change_map.rio.write_nodata(np.nan).rio.to_raster(tif_path)

back_nc = xr.open_dataarray(nc_path, decode_coords="all")
back_tif = rxr.open_rasterio(tif_path, masked=True)
print(nc_path.name, round(nc_path.stat().st_size / 1e6, 2), "MB |", dict(back_nc.sizes), back_nc.rio.crs)
print(tif_path.name, round(tif_path.stat().st_size / 1e6, 2), "MB |", back_tif.shape, back_tif.rio.crs, back_tif.rio.resolution())
print("NetCDF equal:", bool(np.allclose(back_nc, yearly, equal_nan=True)),
      "| GeoTIFF equal:", bool(np.allclose(back_tif.squeeze("band"), change_map, equal_nan=True)))

The opened variable is a `dask.array` with four chunks of 28 dates. `lazy_mean` is still a dask array after the NDVI, the mask and the mean: nothing has been read yet. `.compute()` runs the recipe and returns a NumPy array, equal to the mean of the `ndvi` cube computed without dask.

For 51 MB, dask brings no gain; it adds a little overhead. Its value shows when a cube is larger than memory: the same code then runs chunk by chunk. When you meet cubes of a whole country, from a STAC catalogue, this is how they are handled.

The NetCDF file keeps all five yearly maps with the dimension `year`, the coordinates and the CRS; read back, it equals `yearly`. The GeoTIFF holds one map, the change from 2020 to 2021, with the CRS and 100 m cells, and `NaN` as nodata. Both files are now in `../outputs`, ready for QGIS.

`yearly.name` names the variable inside the NetCDF file, and `.attrs` adds notes that travel with it: the same kind of notes you read in step 1.

*Contains modified Copernicus Sentinel data 2019–2023, processed by ESA.*

### Practice

**Q1 · one line:** How many chunks does `lazy["B08"]` have in total? Use `.data.npartitions`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
lazy["B08"].data.npartitions
```

4: the time dimension is cut into 4 pieces of 28 dates, while `y` and `x` stay whole.
</details>

**Q2 · two lines:** Save the **2021** map of `yearly` as a GeoTIFF `d06_ndvi_oct_dec_2021.tif` in `../outputs` (with `NaN` as nodata), then print the file's size in KB, rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
yearly.sel(year=2021).rio.write_nodata(np.nan).rio.to_raster(OUT / "d06_ndvi_oct_dec_2021.tif")
print(round((OUT / "d06_ndvi_oct_dec_2021.tif").stat().st_size / 1000, 1))
```

About 114.7 KB: 28,482 cells of 4 bytes (`float32`) plus a small header. A GeoTIFF holds one map per band; for a time series, NetCDF is the natural format.
</details>

**Q3 · three lines:** Compute the **number of clear dates** of every cell with dask: `lazy["SCL"].isin([4, 5, 6]).sum(dim="time")`, then `.compute()`. Print the smallest and largest count, and the count at the cell nearest to x = 356,550 m, y = 3,084,850 m.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
clear_count = lazy["SCL"].isin([4, 5, 6]).sum(dim="time").compute()
print(int(clear_count.min()), int(clear_count.max()))
print(int(clear_count.sel(x=356550, y=3084850, method="nearest")))
```

Between 45 and 112 clear dates per cell, and 100 at the valley-floor cell. Some cells were clear on every date, others on fewer than half: every cell has its own time series with its own gaps.
</details>

## 9. When it breaks: a date that is not in the cube

D3's after image was taken on 11 November 2021. You select it from the cube with a `pd.Timestamp`, the pandas date object of A11. Run the cell.

In [ ]:
after_day = cube.sel(time=pd.Timestamp("2021-11-11"))

Read the last line:

`KeyError: "not all values found in index 'time'. Try setting the `method` keyword argument (example: method='nearest')."`

- **KeyError** (A4): a label you asked for is not in the index.
- **not all values found in index 'time'**: the label is a time.
- But 11 November 2021 **is** in the cube! Look at step 1 again: every date carries a clock time. The image is labelled `2021-11-11T05:00:29`, and `pd.Timestamp("2021-11-11")` means midnight. With an exact label, `.sel` wants an exact match.

Two fixes:

- `cube.sel(time="2021-11-11")`: a date **string** selects everything on that day, whatever the clock time. The result keeps a `time` dimension of length 1.
- `cube.sel(time="2021-11-11", method="nearest")`: the date nearest to that moment, without a `time` dimension. Check which date you got, because "nearest" always finds one.

**Quieter traps** do not raise an error at all, which makes them more dangerous. The next cell shows two: a day without an image gives an **empty** selection, and a `y` slice written from south to north selects **nothing**, because `y` runs from north to south. The cure is the same for both: print `.sizes` after every selection. And select by **name**, `.isel(y=50, x=30)`, rather than by position in `.values[:, 50, 30]`: names cannot be mixed up, positions can.

**Try it:** change the cell above to `cube.sel(time="2021-11-11")` and print `.sizes`.

In [ ]:
print("a day with an image:   ", cube.sel(time="2021-11-11").sizes["time"], "date")
print("a day without an image:", cube.sel(time="2021-11-10").sizes["time"], "dates, and no error")
print("y slice north to south:", cube.sel(y=slice(3094000, 3093000)).sizes["y"], "rows")
print("y slice south to north:", cube.sel(y=slice(3093000, 3094000)).sizes["y"], "rows, and no error")

## 10. What this can and cannot tell you

**What it can tell you**

- What a **normal year** looks like on the valley floor and in the forest: a forest green all year, and fields and grass that green up with the monsoon.
- **When** the valley floor changed: between the images of 6 June and 29 July 2021, which fits the June 2021 flood. Two images, as in D3, could only say "between October 2020 and November 2021".
- That the change **lasted**: in the same season of 2022 and 2023, the valley floor's NDVI stayed far below its 2019 and 2020 values, while the forest and the whole box did not change.

**What it cannot tell you**

- **The day.** The monsoon hid the valley for seven weeks around the flood. Only radar satellites, which see through clouds, or reports from the ground could narrow it down.
- **Why NDVI stayed low.** New sediment, a wider river bed, construction, or plants that have not returned all look alike to NDVI (D3).
- **Details smaller than 100 m.** A 100 m cell mixes river, bank, road and field. The strip is often only one or two cells wide here, so its edges and its area are rough.
- **Anything SCL got wrong.** The cloud mask is a model's output. Snow that is cloud, sediment that is "cloud": every mistake removes or keeps the wrong cells (review 1).
- **Anything you smoothed or filled.** A moving average moves sudden events in time, and an interpolated month was never observed.

As in D3, report the method with the result: the dates, the mask, the area, the season you compared and the limits.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · What did SCL call the new sediment?

For the valley-floor cells (`cube["SCL"].where(floor)`), compare two periods: **October–December 2020** and **July 2021–January 2022**. For each, print the percentage of valley-floor observations with codes 4, 5 or 6 (clear), with 7 (unclassified) and with 8, 9 or 10 (cloud). Then print the same cloud percentage for the **whole box** in the second period.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
floor_scl = cube["SCL"].where(floor)
for label, period in [("Oct-Dec 2020", slice("2020-10", "2020-12")), ("Jul 2021-Jan 2022", slice("2021-07", "2022-01"))]:
    scl_period = floor_scl.sel(time=period)
    n_obs = int(scl_period.count())
    for name, codes in [("clear", [4, 5, 6]), ("unclassified", [7]), ("cloud", [8, 9, 10])]:
        print(label, name, round(int(scl_period.isin(codes).sum()) / n_obs * 100, 1), "%")

whole = cube["SCL"].sel(time=slice("2021-07", "2022-01"))
print("whole box, cloud:", round(float(whole.isin([8, 9, 10]).mean()) * 100, 1), "%")
```

In October–December 2020, 98.4% of the valley-floor observations were clear and practically none (0.0%) were called cloud or unclassified. From July 2021 to January 2022, only 39.7% were clear: 21.7% were *unclassified* and 36.6% *cloud*, while the whole box was only about 7.4% cloud in the same months. The fresh, bright sediment confused ESA's processor: it looks more like cloud than like any ground it knows. So the cloud mask removed a large part of the new sediment, exactly where the change happened. The valley floor's NDVI after the flood rests on the cells the mask kept. Masks are models too; check what they remove where it matters. `.where(floor)` turns all other cells into `NaN`; `.isin` counts `NaN` as False, and `.count()` skips it.
</details>

### Review 2 · A second index: NIR against short-wave infrared

The normalised difference of B08 and B11, (B08 − B11) / (B08 + B11), is often called **NDMI** (normalised difference moisture index): it is high for green, moist leaves and low for bare, dry ground. Compute it for every clear date, take the median for the valley floor and the forest box, and plot both series with dates on the x axis. Does it tell the same story as NDVI?

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
ndmi = ((refl["B08"] - refl["B11"]) / (refl["B08"] + refl["B11"])).where(clear)
floor_ndmi = ndmi.where(floor).median(dim=["x", "y"])
forest_ndmi = ndmi.sel(x=slice(358500, 359500), y=slice(3094000, 3093000)).median(dim=["x", "y"])
print("valley floor, Oct-Dec 2020:", round(float(floor_ndmi.sel(time=slice("2020-10", "2020-12")).median()), 3),
      "| Oct-Dec 2021:", round(float(floor_ndmi.sel(time=slice("2021-10", "2021-12")).median()), 3))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(forest_ndmi["time"], forest_ndmi, marker=".", color="darkgreen", label="forest box")
ax.plot(floor_ndmi["time"], floor_ndmi, marker=".", color="saddlebrown", label="valley floor")
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_ylabel("Median NDMI (B08, B11)")
ax.legend()
plt.show()
```

About 0.01 in October–December 2020 and −0.077 a year later. The chart tells the same story as NDVI: the forest stays high (about 0.15 to 0.35), and the valley floor drops in mid-2021 and stays low. Before the flood, the valley floor's NDMI swings strongly with the seasons, from below 0 in winter to almost the forest's values at the end of the monsoon, when the fields are green and moist. Two indices from different bands agreeing is good evidence (D3); and B11 was not used to choose the valley floor.
</details>

### Review 3 · When is each cell greenest?

Build a monthly climatology of the whole NDVI cube with `ndvi.groupby("time.month").median()`, then find for every cell the month of its highest median with `.idxmax("month")`. Print how many cells peak in each month, and map the peak month with a colour map that has 12 steps (`plt.get_cmap("twilight", 12)`).

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
climatology = ndvi.groupby("time.month").median()
peak_month = climatology.idxmax("month")
months, n_peak = np.unique(peak_month.values, return_counts=True)
print("whole box:", dict(zip(months.astype(int).tolist(), n_peak.tolist())))
floor_months, floor_n = np.unique(peak_month.values[floor.values], return_counts=True)
print("valley floor:", dict(zip(floor_months.astype(int).tolist(), floor_n.tolist())))

fig, ax = plt.subplots(figsize=(6, 8))
im = ax.imshow(peak_month, extent=extent_km, cmap=plt.get_cmap("twilight", 12), vmin=0.5, vmax=12.5)
fig.colorbar(im, ax=ax, shrink=0.6, ticks=range(1, 13), label="Month of the highest median NDVI")
ax.set_title("When is each cell greenest? (2019-2023)")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.show()
```

Most of the valley peaks in October (11,046 cells) or September (10,830): after the monsoon, as the forest did in step 5. The valley floor is different: most of its cells peak in **August (122 of 283) or June (93)**, and the river shows up on the map as a dark thread. That is not a season. A climatology over 2019–2023 mixes years before and after the flood. In June, 6 of the 10 dates come from before the flood (2019 to early June 2021); in August 4 of 8, and many post-flood August dates are masked on the valley floor (review 1); in October only 4 of 10. Where most of a month's clear dates come from before the flood, the median is a pre-flood value, and that month "peaks". A typical year only makes sense for a period in which nothing changed for good. `twilight` is a **cyclic** colour map: December and January get similar colours, like neighbouring months.
</details>

### Review 4 · A time series anywhere, as a table

Write a function `ndvi_series(lon, lat)` that turns a WGS84 position into UTM 45N (B4) and returns the NDVI series of the nearest cell as a pandas Series. Use it for Melamchi town (85.5761° E, 27.8293° N, D5) and for the centre of the forest box (85.5666° E, 27.9591° N), put both into one DataFrame with the dates as rows, save it as `d06_ndvi_two_places.csv` in `../outputs`, and print the median of each column for 2020 and for 2022.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
def ndvi_series(lon, lat):
    """NDVI of the cube cell nearest to a WGS84 position, as a pandas Series."""
    point = gpd.GeoSeries(gpd.points_from_xy([lon], [lat]), crs="EPSG:4326").to_crs("EPSG:32645")
    return ndvi.sel(x=point.x.iloc[0], y=point.y.iloc[0], method="nearest").to_pandas().astype(float)


places = pd.DataFrame({"melamchi_town": ndvi_series(85.5761, 27.8293), "forest_box": ndvi_series(85.5666, 27.9591)})
places.to_csv(OUT / "d06_ndvi_two_places.csv")
print("2020:", places.loc["2020"].median().round(3).to_dict())
print("2022:", places.loc["2022"].median().round(3).to_dict())
```

In 2020 the median NDVI of Melamchi town's cell was about 0.384, in 2022 about 0.305; the forest cell stayed high (about 0.753 and 0.788). The town cell mixes houses, fields and the river bed, so its fall is much smaller than the valley floor's. One function now gives the time series of any place you can name in longitude and latitude, and the CSV file opens in any spreadsheet. `.astype(float)` turns the `float32` values into ordinary Python numbers, so they print cleanly.
</details>

## Recap

- A **data cube** stacks the same grid over time: dimensions `time`, `y`, `x`. `xr.open_dataset(..., decode_coords="all")` opens a NetCDF cube with its CRS; **mask and scale** turns the no-data value 0 into `NaN` and the values into `float32`. Read `.sizes`, `.data_vars`, `.attrs` and the encoding first.
- Select by label with `.sel`: a date string selects a whole day, month or year; `slice` a period; `method="nearest"` one date or one cell (check what you got). Select by position with `.isel`, always by **name**.
- Mask clouds with **SCL** (`isin([4, 5, 6])` and `.where`), and remember that SCL is a model: it probably called monsoon cloud snow, and it called much of the fresh sediment cloud. NDVI for every date is one line; `.median(dim=["x", "y"])` turns an area into a time series.
- `groupby("time.month")` gives the typical year (climatology); `resample(time="1MS")` regular months; `rolling` smooths but blurs sudden events; `interpolate_na` invents values. Compare **the same season** across years.
- The valley floor's NDVI fell between 6 June and 29 July 2021 and stayed low in 2022 and 2023, while the forest did not change. **dask** (`chunks=`, `.compute()`) handles cubes larger than memory; NetCDF stores cubes, GeoTIFF single maps.

### Words you learned

| Word | Meaning |
| --- | --- |
| data cube | The same grid of cells repeated over time (and sometimes over bands): dimensions such as time, y and x |
| dimension | A named axis of an array, such as `time`, `y` or `x` |
| Dataset / DataArray | In xarray: several variables that share dimensions / one variable with its dimensions and coordinates |
| data variable | One of the arrays inside a Dataset, such as `B08` or `SCL` |
| NetCDF | A file format for multi-dimensional arrays with their coordinates and notes, common for cubes |
| mask and scale | What xarray does while reading: no-data values become `NaN`, and a stored scale factor is applied |
| encoding | How a variable is stored on disk (data type, no-data value, compression), kept in `.encoding` |
| STAC | SpatioTemporal Asset Catalog: a standard way to search and load satellite images by place, time and cloud cover |
| time series | Values of one place, or one area, ordered by date |
| scene classification (SCL) | A Sentinel-2 Level-2A layer that labels each cell as vegetation, cloud, water, snow and so on |
| cloud mask | A True/False grid that removes cells covered by cloud or its shadow |
| monsoon gap | Months with few or no clear images because of monsoon clouds |
| climatology | The typical year: the median (or mean) of all Januaries, all Februaries, and so on |
| anomaly | A value minus the typical value for its month |
| composite (in time) | One value per cell made from several dates, such as a monthly median |
| moving average | The mean of a value and its neighbours in time, used to smooth a series |
| interpolation (in time) | Filling a missing date with values estimated from the dates around it |
| change map | One map minus another map of the same place, cell by cell |
| dask / chunk | A library that works on arrays in pieces, in parallel and only when asked / one such piece |
| lazy loading | Reading data only when a result is really needed, with `.compute()` |

## Next

**This is the end of the PyGILE Course.** Congratulations! You started with `print()` and a calculator in A1. You now read and map vector data, analyse terrain, process satellite images, measure change and work with five years of data in one cube, all on real data from Nepal. That is a real set of skills, and you built it one question at a time.

Keep going with **[PyGILE-Plus](https://github.com/Geoinformatics-Lab/PyGILE-Plus)**. It builds on everything here with landslide susceptibility modelling, deep learning and more tools (SAGA, GRASS, Orfeo ToolBox) in one ready-made Docker image. And try your new skills on a place you know: your own district, your own river, your own question. Thank you for learning with PyGILE.

<!-- pygile-nav -->
**PyGILE Course** · lesson 39 of 39 · [Course home](../README.md)  
← Previous: [D5 · Roads and networks: what the flood cut](D05_Roads_and_networks.ipynb) · You have reached the end of the course